# 01 — Data Audit and Governance

## Plate Value Intelligence

This notebook validates the structure, coverage, completeness, and modelling readiness of the 2025 DVLA auction dataset.

### Objectives

- Load and inspect the raw auction data
- Validate row and column structure
- Review event-level coverage
- Identify missing and duplicate records
- Assess hammer-price completeness
- Add governance and data-quality fields
- Produce a clean core dataset for valuation modelling

### Modelling principle

The 2025 full-auction dataset will be used as the core modelling dataset. Historical partial and editorial datasets will not be mixed into model training.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

/Users/osmanorka/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/matplotlib-zzlvmx6x because there was an issue with the default path ({configdir}); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [2]:
PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_FILE = RAW_DATA_DIR / "dvla_auction_results_2025.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file: {RAW_FILE}")
print(f"File exists: {RAW_FILE.exists()}")

Project root: /Users/osmanorka/Plate-Value-Intelligence
Raw file: /Users/osmanorka/Plate-Value-Intelligence/data/raw/dvla_auction_results_2025.csv
File exists: True


In [3]:
df_raw = pd.read_csv(RAW_FILE)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

Rows: 18,000
Columns: 9


## 1. Initial Data Inspection

First, we inspect the dataset structure, sample records, column names, and data types before applying any transformations.

In [4]:
df_raw.head()

Out[0]: 
   auction_year auction_month event_code  lot_number     plate  \
0          2025       January       B270           1    1984 A   
1          2025       January       B270           2   7777 AA   
2          2025       January       B270           3    AAD 8L   
3          2025       January       B270           4  A881 AKA   
4          2025       January       B270           5   AAK 64R   

   hammer_price_gbp  \
0          6,510.00   
1          5,560.00   
2          3,710.00   
3            480.00   
4          3,140.00   

                                                      source_url collected_at  \
0  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
1  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
2  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
3  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
4  https://dvlaauction.co.uk/Event/DownloadExcel?E

In [5]:
print("Column names:")
for column in df_raw.columns:
    print(f"- {column}")

print("\nDataset information:")
df_raw.info()

Column names:
- auction_year
- auction_month
- event_code
- lot_number
- plate
- hammer_price_gbp
- source_url
- collected_at
- source

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   auction_year      18000 non-null  int64  
 1   auction_month     18000 non-null  str    
 2   event_code        18000 non-null  str    
 3   lot_number        18000 non-null  int64  
 4   plate             18000 non-null  str    
 5   hammer_price_gbp  17782 non-null  float64
 6   source_url        18000 non-null  str    
 7   collected_at      18000 non-null  str    
 8   source            18000 non-null  str    
dtypes: float64(1), int64(2), str(6)
memory usage: 1.2 MB


In [6]:
df_raw.sample(10, random_state=42)

Out[0]: 
       auction_year auction_month event_code  lot_number     plate  \
2574           2025      February       B271         575    10 ESC   
7496           2025           May       B273        1497  R244 NGE   
9210           2025          June       B274        1211   M78 MHK   
5456           2025         March       B272        1457   R53 BEP   
736            2025       January       B270         737   G80 YYH   
11770          2025          July       B275        1771  TCR 805S   
856            2025       January       B270         857   JAS 14S   
7273           2025           May       B273        1274     8 NJL   
11499          2025          July       B275        1500  RIG 8055   
11605          2025          July       B275        1606  S172 CYL   

       hammer_price_gbp  \
2574           6,010.00   
7496           1,650.00   
9210             200.00   
5456             910.00   
736              810.00   
11770          1,010.00   
856            1,760.00   
7273

## 2. Missing-Value Assessment

Missing values are reviewed before modelling. A missing hammer price will not automatically be interpreted as an unsold plate because the underlying reason has not yet been verified.

In [7]:
missing_summary = (
    df_raw.isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df_raw) * 100
)

missing_summary.sort_values(
    "missing_count",
    ascending=False
)

Out[0]: 
                  missing_count  missing_percentage
hammer_price_gbp            218                1.21
auction_year                  0                0.00
auction_month                 0                0.00
event_code                    0                0.00
lot_number                    0                0.00
plate                         0                0.00
source_url                    0                0.00
collected_at                  0                0.00
source                        0                0.00


## 3. Duplicate Record Assessment

Duplicate checks are performed at both full-row and business-key levels. In this dataset, the expected business key is the combination of auction event and lot number.

In [8]:
full_row_duplicates = df_raw.duplicated().sum()

event_lot_duplicates = df_raw.duplicated(
    subset=["event_code", "lot_number"],
    keep=False
).sum()

print(f"Full-row duplicates: {full_row_duplicates:,}")
print(f"Duplicate event-lot records: {event_lot_duplicates:,}")

Full-row duplicates: 0
Duplicate event-lot records: 0


## 4. Auction Event Coverage

Event-level coverage is assessed to confirm the number of lots, lot-number ranges, and price completeness within each auction event.

In [9]:
event_coverage = (
    df_raw.groupby(
        ["auction_year", "auction_month", "event_code"],
        dropna=False
    )
    .agg(
        total_lots=("lot_number", "size"),
        minimum_lot=("lot_number", "min"),
        maximum_lot=("lot_number", "max"),
        unique_lots=("lot_number", "nunique"),
        prices_recorded=("hammer_price_gbp", "count"),
        minimum_price=("hammer_price_gbp", "min"),
        median_price=("hammer_price_gbp", "median"),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
)

event_coverage["prices_not_recorded"] = (
    event_coverage["total_lots"]
    - event_coverage["prices_recorded"]
)

event_coverage

Out[0]: 
   auction_year auction_month event_code  total_lots  minimum_lot  \
0          2025      February       B271        2000            1   
1          2025       January       B270        2000            1   
2          2025          July       B275        2000            1   
3          2025          June       B274        2000            1   
4          2025         March       B272        2000            1   
5          2025           May       B273        2000            1   
6          2025      November       B278        2000            1   
7          2025       October       B277        2000            1   
8          2025     September       B276        2000            1   

   maximum_lot  unique_lots  prices_recorded  minimum_price  median_price  \
0         2000         2000             1969         200.00      1,510.00   
1         2000         2000             1966         200.00      1,485.00   
2         2000         2000             1970         200.00      1,51

## 5. Lot Sequence Validation

Each event is expected to contain a unique and continuous lot-number sequence. Missing or unexpected lot numbers may indicate incomplete extraction.

In [10]:
lot_sequence_checks = []

for event_code, event_df in df_raw.groupby("event_code"):
    observed_lots = set(event_df["lot_number"].dropna().astype(int))
    
    expected_lots = set(
        range(
            int(event_df["lot_number"].min()),
            int(event_df["lot_number"].max()) + 1
        )
    )
    
    missing_lots = sorted(expected_lots - observed_lots)
    
    lot_sequence_checks.append(
        {
            "event_code": event_code,
            "observed_lot_count": len(observed_lots),
            "expected_lot_count": len(expected_lots),
            "missing_lot_count": len(missing_lots),
            "missing_lot_examples": missing_lots[:10],
        }
    )

lot_sequence_summary = pd.DataFrame(lot_sequence_checks)
lot_sequence_summary

Out[0]: 
  event_code  observed_lot_count  expected_lot_count  missing_lot_count  \
0       B270                2000                2000                  0   
1       B271                2000                2000                  0   
2       B272                2000                2000                  0   
3       B273                2000                2000                  0   
4       B274                2000                2000                  0   
5       B275                2000                2000                  0   
6       B276                2000                2000                  0   
7       B277                2000                2000                  0   
8       B278                2000                2000                  0   

  missing_lot_examples  
0                   []  
1                   []  
2                   []  
3                   []  
4                   []  
5                   []  
6                   []  
7                   []  
8              

## 6. Hammer-Price Validity Assessment

Recorded hammer prices are checked for invalid, zero, negative, and unusually extreme values before modelling.

In [11]:
price_quality_summary = pd.Series(
    {
        "total_rows": len(df_raw),
        "price_recorded": df_raw["hammer_price_gbp"].notna().sum(),
        "price_not_recorded": df_raw["hammer_price_gbp"].isna().sum(),
        "zero_prices": df_raw["hammer_price_gbp"].eq(0).sum(),
        "negative_prices": df_raw["hammer_price_gbp"].lt(0).sum(),
        "minimum_recorded_price": df_raw["hammer_price_gbp"].min(),
        "median_recorded_price": df_raw["hammer_price_gbp"].median(),
        "maximum_recorded_price": df_raw["hammer_price_gbp"].max(),
    },
    name="value"
)

price_quality_summary

Out[0]: 
total_rows                18,000.00
price_recorded            17,782.00
price_not_recorded           218.00
zero_prices                    0.00
negative_prices                0.00
minimum_recorded_price       200.00
median_recorded_price      1,510.00
maximum_recorded_price   102,010.00
Name: value, dtype: float64


## 7. Governance and Data-Quality Classification

Governance fields are added to preserve source lineage, coverage status, price provenance, modelling eligibility, and the observed availability of hammer-price information.

In [12]:
df_governed = df_raw.copy()

df_governed["data_source"] = "dvlaauction"
df_governed["coverage_level"] = "full"
df_governed["price_basis"] = "direct_hammer"
df_governed["source_reliability"] = "high"
df_governed["selection_bias"] = "full_event_catalogue"
df_governed["model_eligibility"] = "core"

df_governed["price_record_status"] = np.where(
    df_governed["hammer_price_gbp"].notna(),
    "price_recorded",
    "price_not_recorded"
)

df_governed.head()

Out[0]: 
   auction_year auction_month event_code  lot_number     plate  \
0          2025       January       B270           1    1984 A   
1          2025       January       B270           2   7777 AA   
2          2025       January       B270           3    AAD 8L   
3          2025       January       B270           4  A881 AKA   
4          2025       January       B270           5   AAK 64R   

   hammer_price_gbp  \
0          6,510.00   
1          5,560.00   
2          3,710.00   
3            480.00   
4          3,140.00   

                                                      source_url collected_at  \
0  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
1  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
2  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
3  https://dvlaauction.co.uk/Event/DownloadExcel?EventID=4377902   2026-06-20   
4  https://dvlaauction.co.uk/Event/DownloadExcel?E

## 8. Core Valuation Dataset

Only records with a directly observed hammer price are eligible for supervised valuation modelling. Records without a recorded price remain in the governed dataset but are excluded from the modelling target.

In [13]:
valuation_core_2025 = (
    df_governed[
        df_governed["hammer_price_gbp"].notna()
    ]
    .copy()
)

print(f"Governed rows: {len(df_governed):,}")
print(f"Valuation modelling rows: {len(valuation_core_2025):,}")
print(
    f"Excluded from supervised valuation: "
    f"{len(df_governed) - len(valuation_core_2025):,}"
)

Governed rows: 18,000
Valuation modelling rows: 17,782
Excluded from supervised valuation: 218


## 9. Automated Data-Quality Checks

Explicit assertions are used to ensure that critical structural assumptions remain valid whenever the notebook is rerun.

In [14]:
assert len(df_raw) == 18_000, "Unexpected total row count."
assert df_raw["event_code"].nunique() == 9, "Unexpected event count."
assert full_row_duplicates == 0, "Full-row duplicates detected."
assert event_lot_duplicates == 0, "Duplicate event-lot keys detected."
assert lot_sequence_summary["missing_lot_count"].sum() == 0, (
    "Missing lot numbers detected."
)
assert df_raw["hammer_price_gbp"].dropna().gt(0).all(), (
    "Non-positive hammer prices detected."
)

print("All critical data-quality checks passed.")

All critical data-quality checks passed.


In [15]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

Processed data directory: /Users/osmanorka/Plate-Value-Intelligence/data/processed
Reports directory: /Users/osmanorka/Plate-Value-Intelligence/reports


## 10. Governed Dataset Outputs

The complete governed dataset and the price-observed valuation core are exported separately to preserve both auditability and modelling readiness.

In [16]:
governed_output = (
    PROCESSED_DATA_DIR
    / "all_sources_governed_2025.csv"
)

valuation_output = (
    PROCESSED_DATA_DIR
    / "valuation_core_2025.csv"
)

df_governed.to_csv(
    governed_output,
    index=False
)

valuation_core_2025.to_csv(
    valuation_output,
    index=False
)

print(f"Saved: {governed_output}")
print(f"Saved: {valuation_output}")

Saved: /Users/osmanorka/Plate-Value-Intelligence/data/processed/all_sources_governed_2025.csv
Saved: /Users/osmanorka/Plate-Value-Intelligence/data/processed/valuation_core_2025.csv


## 11. Data-Quality Summary

A concise quality summary is produced for technical review, governance reporting, and stakeholder communication.

In [17]:
data_quality_summary = pd.DataFrame(
    {
        "metric": [
            "Total records",
            "Auction events",
            "Unique event-lot keys",
            "Full-row duplicates",
            "Duplicate event-lot keys",
            "Missing lot numbers",
            "Recorded hammer prices",
            "Prices not recorded",
            "Price completeness",
            "Zero prices",
            "Negative prices",
            "Minimum recorded price",
            "Median recorded price",
            "Maximum recorded price",
        ],
        "value": [
            len(df_governed),
            df_governed["event_code"].nunique(),
            df_governed[["event_code", "lot_number"]]
            .drop_duplicates()
            .shape[0],
            full_row_duplicates,
            event_lot_duplicates,
            lot_sequence_summary["missing_lot_count"].sum(),
            df_governed["hammer_price_gbp"].notna().sum(),
            df_governed["hammer_price_gbp"].isna().sum(),
            (
                df_governed["hammer_price_gbp"].notna().mean()
                * 100
            ),
            df_governed["hammer_price_gbp"].eq(0).sum(),
            df_governed["hammer_price_gbp"].lt(0).sum(),
            df_governed["hammer_price_gbp"].min(),
            df_governed["hammer_price_gbp"].median(),
            df_governed["hammer_price_gbp"].max(),
        ],
    }
)

data_quality_summary

Out[0]: 
                      metric      value
0              Total records  18,000.00
1             Auction events       9.00
2      Unique event-lot keys  18,000.00
3        Full-row duplicates       0.00
4   Duplicate event-lot keys       0.00
5        Missing lot numbers       0.00
6     Recorded hammer prices  17,782.00
7        Prices not recorded     218.00
8         Price completeness      98.79
9                Zero prices       0.00
10           Negative prices       0.00
11    Minimum recorded price     200.00
12     Median recorded price   1,510.00
13    Maximum recorded price 102,010.00


## 12. Event-Level Price Completeness

Price completeness is compared across auction events to identify whether missing price records are concentrated in specific periods.

In [18]:
event_price_completeness = (
    df_governed.groupby(
        ["auction_month", "event_code"],
        as_index=False
    )
    .agg(
        total_lots=("lot_number", "size"),
        prices_recorded=("hammer_price_gbp", "count"),
    )
)

event_price_completeness["prices_not_recorded"] = (
    event_price_completeness["total_lots"]
    - event_price_completeness["prices_recorded"]
)

event_price_completeness["price_completeness_pct"] = (
    event_price_completeness["prices_recorded"]
    / event_price_completeness["total_lots"]
    * 100
)

event_price_completeness

Out[0]: 
  auction_month event_code  total_lots  prices_recorded  prices_not_recorded  \
0      February       B271        2000             1969                   31   
1       January       B270        2000             1966                   34   
2          July       B275        2000             1970                   30   
3          June       B274        2000             1981                   19   
4         March       B272        2000             1983                   17   
5           May       B273        2000             1983                   17   
6      November       B278        2000             1981                   19   
7       October       B277        2000             1977                   23   
8     September       B276        2000             1972                   28   

   price_completeness_pct  
0                   98.45  
1                   98.30  
2                   98.50  
3                   99.05  
4                   99.15  
5                   99

In [19]:
plot_data = event_price_completeness.set_index(
    "event_code"
)["price_completeness_pct"]

ax = plot_data.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title(
    "Hammer-Price Completeness by Auction Event"
)
ax.set_xlabel("Auction Event")
ax.set_ylabel("Price Completeness (%)")
ax.set_ylim(95, 100)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

Fallback to a different backend
<ipython-input-1-6ad4fce979e0>:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
recorded_prices = valuation_core_2025[
    "hammer_price_gbp"
]

ax = recorded_prices.plot(
    kind="hist",
    bins=60,
    figsize=(10, 5)
)

ax.set_title("Distribution of Recorded Hammer Prices")
ax.set_xlabel("Hammer Price (£)")
ax.set_ylabel("Number of Plates")

plt.tight_layout()
plt.show()

<ipython-input-1-c1eef2cec896>:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
log_prices = np.log1p(recorded_prices)

ax = log_prices.plot(
    kind="hist",
    bins=60,
    figsize=(10, 5)
)

ax.set_title(
    "Distribution of Log-Transformed Hammer Prices"
)
ax.set_xlabel("log(1 + Hammer Price)")
ax.set_ylabel("Number of Plates")

plt.tight_layout()
plt.show()

<ipython-input-1-6ea3b90dc603>:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
price_quantiles = (
    recorded_prices.quantile(
        [
            0.00,
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
    .rename("hammer_price_gbp")
    .to_frame()
)

price_quantiles

Out[0]: 
      hammer_price_gbp
0.00            200.00
0.01            200.00
0.05            230.00
0.10            320.00
0.25            730.00
0.50          1,510.00
0.75          3,110.00
0.90          5,690.00
0.95          7,849.00
0.99         14,020.00
1.00        102,010.00


In [23]:
data_quality_output = (
    REPORTS_DIR
    / "data_quality_summary_2025.csv"
)

event_coverage_output = (
    REPORTS_DIR
    / "event_coverage_summary_2025.csv"
)

price_quantiles_output = (
    REPORTS_DIR
    / "price_quantiles_2025.csv"
)

data_quality_summary.to_csv(
    data_quality_output,
    index=False
)

event_coverage.to_csv(
    event_coverage_output,
    index=False
)

price_quantiles.to_csv(
    price_quantiles_output
)

print(f"Saved: {data_quality_output}")
print(f"Saved: {event_coverage_output}")
print(f"Saved: {price_quantiles_output}")

Saved: /Users/osmanorka/Plate-Value-Intelligence/reports/data_quality_summary_2025.csv
Saved: /Users/osmanorka/Plate-Value-Intelligence/reports/event_coverage_summary_2025.csv
Saved: /Users/osmanorka/Plate-Value-Intelligence/reports/price_quantiles_2025.csv


## 14. Audit Conclusion

The 2025 DVLA auction dataset contains 18,000 lots across nine complete auction events.

Key findings:

- Each event contains a complete and continuous sequence of 2,000 unique lots.
- No full-row or event-lot duplicate records were identified.
- Hammer prices were recorded for 17,782 lots.
- A total of 218 lots have no recorded hammer price and remain classified as `price_not_recorded`.
- No zero or negative hammer prices were observed.
- Recorded prices range from £200 to £102,010, with a median of £1,510.
- The price distribution is strongly right-skewed, supporting the later evaluation of a log-transformed modelling target.
- The dataset is considered suitable as the core source for valuation modelling, premium-asset analysis, and event-based validation.

The next notebook will transform licence-plate text into explainable, commercially meaningful modelling features.